# Ultrasound Segmentation — U-Net Training
EfficientNet-B0 encoder · Combined BCE + Dice loss · Metrics: Dice, Pixel Accuracy

## 1. Imports & Device

In [ ]:
import torch
import torch.optim as optim
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from data_loader import load_ultrasound_data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Metrics

In [ ]:
def calculate_dice(preds, targets, smooth=1e-6):
    preds = preds.view(-1)
    targets = targets.view(-1)
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)


def calculate_pixel_accuracy(preds, targets):
    correct = (preds == targets).float().sum()
    total = targets.numel()
    return correct / total

## 3. Model

In [ ]:
def get_model():
    model = smp.Unet(
        encoder_name="efficientnet-b0",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1
    ).to(device)

    for param in model.parameters():
        param.requires_grad = True

    return model

## 4. Loss

In [ ]:
bce_loss  = torch.nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode='binary')

def combined_loss(outputs, targets):
    return 0.5 * bce_loss(outputs, targets) + 0.5 * dice_loss(outputs, targets)

## 5. Training

In [ ]:
# Metric history — populated by train_model()
history = {
    'loss':      [],
    'train_dice':  [],
    'train_pa':    [],
    'val_dice':    [],
    'val_pa':      [],
}


def train_model(model, train_loader, val_loader, epochs=50):
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    best_dice = 0.0

    print(f'--- Training on {device} ---')

    for epoch in range(epochs):

        # ── TRAIN ──────────────────────────────────────────────────────────
        model.train()
        train_loss = train_dice = train_pa = 0.0

        for batch in train_loader:
            inputs = batch['image'].to(device)
            labels = batch['mask'].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = combined_loss(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_loss += loss.item()
            train_dice += calculate_dice(preds, labels).item()
            train_pa   += calculate_pixel_accuracy(preds, labels).item()

        n = len(train_loader)
        train_loss /= n
        train_dice /= n
        train_pa   /= n

        # ── VALIDATION ─────────────────────────────────────────────────────
        model.eval()
        val_dice = val_pa = 0.0

        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['image'].to(device)
                labels = batch['mask'].to(device)
                outputs = model(inputs)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                val_dice += calculate_dice(preds, labels).item()
                val_pa   += calculate_pixel_accuracy(preds, labels).item()

        m = len(val_loader)
        val_dice /= m
        val_pa   /= m

        # ── RECORD ─────────────────────────────────────────────────────────
        history['loss'].append(train_loss)
        history['train_dice'].append(train_dice)
        history['train_pa'].append(train_pa)
        history['val_dice'].append(val_dice)
        history['val_pa'].append(val_pa)

        # ── LOG ────────────────────────────────────────────────────────────
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'  Train  →  Loss: {train_loss:.4f} | Dice: {train_dice:.4f} | PA: {train_pa:.4f}')
        print(f'  Val    →  Dice: {val_dice:.4f} | PA: {val_pa:.4f}')
        print('-' * 50)

        # ── CHECKPOINT ─────────────────────────────────────────────────────
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), 'best_model.pth')
            print('  Saved new best model!')

## 6. Training Curves
Run after `train_model()` — each metric gets its own figure.

In [ ]:
def plot_training_curves(history):
    epochs = range(1, len(history['loss']) + 1)

    # ── Figure 1: Loss ──────────────────────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(epochs, history['loss'], color='steelblue', linewidth=1.8)
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)
    fig1.tight_layout()
    plt.show()

    # ── Figure 2: Dice ──────────────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.plot(epochs, history['train_dice'], label='Train', color='steelblue', linewidth=1.8)
    ax2.plot(epochs, history['val_dice'],   label='Val',   color='darkorange', linewidth=1.8, linestyle='--')
    ax2.set_title('Dice Score')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()
    plt.show()

    # ── Figure 3: Pixel Accuracy ────────────────────────────────────────────
    fig3, ax3 = plt.subplots(figsize=(8, 4))
    ax3.plot(epochs, history['train_pa'], label='Train', color='steelblue', linewidth=1.8)
    ax3.plot(epochs, history['val_pa'],   label='Val',   color='darkorange', linewidth=1.8, linestyle='--')
    ax3.set_title('Pixel Accuracy')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Pixel Accuracy')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    fig3.tight_layout()
    plt.show()

## 7. Evaluation & Visualisation

In [ ]:
def evaluate_and_visualize(model, loader):
    model.eval()

    all_imgs, all_masks, all_preds = [], [], []
    all_dices, all_pixel_accs = [], []

    print('\n' + '=' * 50)
    print(f'{"Sample #":<10} | {"Dice":<10} | {"Pixel Acc":<10}')
    print('-' * 40)

    with torch.no_grad():
        count = 0
        for batch in loader:
            images = batch['image'].to(device)
            masks  = batch['mask'].to(device)
            outputs = model(images)
            preds  = (torch.sigmoid(outputs) > 0.5).float()

            for i in range(images.size(0)):
                count += 1
                dice = calculate_dice(preds[i], masks[i]).item()
                pa   = calculate_pixel_accuracy(preds[i], masks[i]).item()

                print(f'Image {count:02d}   | {dice:<10.4f} | {pa:<10.4f}')

                all_imgs.append(images[i].cpu())
                all_masks.append(masks[i].cpu())
                all_preds.append(preds[i].cpu())
                all_dices.append(dice)
                all_pixel_accs.append(pa)

    mean_dice = sum(all_dices) / len(all_dices)
    mean_pa   = sum(all_pixel_accs) / len(all_pixel_accs)
    print('-' * 40)
    print(f'MEAN       | {mean_dice:<10.4f} | {mean_pa:<10.4f}\n')

    # ── One figure per sample ───────────────────────────────────────────────
    for i in range(len(all_imgs)):
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        fig.suptitle(
            f'Sample {i+1}  |  Dice: {all_dices[i]:.4f}  |  Pixel Acc: {all_pixel_accs[i]:.4f}',
            fontsize=13
        )

        img = all_imgs[i].permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        axes[0].imshow(img)
        axes[0].set_title('Image')

        axes[1].imshow(all_masks[i].squeeze(), cmap='gray')
        axes[1].set_title('Ground Truth')

        axes[2].imshow(all_preds[i].squeeze(), cmap='viridis')
        axes[2].set_title('Prediction')

        for ax in axes:
            ax.axis('off')

        fig.tight_layout()
        plt.show()

## 8. Run Everything

In [ ]:
train_loader, test_loader = load_ultrasound_data(
    images_dir='../images',
    masks_dir='../masks',
    batch_size=4,
)
model = get_model()

In [ ]:
train_model(model, train_loader, val_loader, epochs=50)

In [ ]:
plot_training_curves(history)

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
print('Loaded best model.')

print('\nEvaluating on validation set:')
evaluate_and_visualize(model, val_loader)